# Carga dos Dados


> Criação da Estrutura Lógica do Data Lake do IMDb em Arquitetura Medalhão

Aqui são criados os schemas:
- Bronze: armazena os dados brutos extraídos da fonte;
- Silver: onde ocorre o tratamento dos dados;
- Gold: armazena os dados refinados e modelados.

In [0]:
CATALOG = "workspace"

VOLUME_PATH = "/Volumes/workspace/default/imdb_mvp"

# Schemas da arquitetura medalhão
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

DataFrame[]

> Função de ingestão de dados para a camada Bronze

Esta célula define a função ingest_bronze(), que lê os arquivos localizados em VOLUME_PATH. Então, os dados são salvos numa tabela Delta no schema workspace.bronze.


In [0]:
from pyspark.sql import functions as F

def ingest_bronze(file_name: str, table_name: str):

    df = (
        spark.read
        .option("header", True)
        .option("sep", "\t")
        .option("nullValue", "\\N")
        .option("inferSchema", False)
        .csv(f"{VOLUME_PATH}/{file_name}")
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file", F.lit(file_name))
    )

    full_table = f"workspace.bronze.{table_name}"

    df.write.mode("overwrite") \
      .format("delta") \
      .saveAsTable(full_table)

    count = spark.table(full_table).count()

    print(f"{full_table}: {count:,} linhas gravadas")

    return df

> Execução da ingestão dos arquivos do IMDb para a camada bronze

Ingestão dos arquivos baixados do IMDb para a camada Bronze, criando as tabelas title_basics, title_ratings, title_principals, name_basics e title_episode.

In [0]:
df_title_basics   = ingest_bronze("title.basics.tsv.gz",     "title_basics")
df_title_ratings  = ingest_bronze("title.ratings.tsv.gz",    "title_ratings")
df_title_principals = ingest_bronze("title.principals.tsv.gz", "title_principals")
df_name_basics    = ingest_bronze("name.basics.tsv.gz",      "name_basics")
df_title_episode  = ingest_bronze("title.episode.tsv.gz",    "title_episode")

workspace.bronze.title_basics: 12,817,267 linhas gravadas
workspace.bronze.title_ratings: 1,714,406 linhas gravadas
workspace.bronze.title_principals: 101,946,601 linhas gravadas
workspace.bronze.name_basics: 15,687,583 linhas gravadas
workspace.bronze.title_episode: 9,911,000 linhas gravadas


> Exibição do esquema das tabelas da camada Bronze

Percorre todas as tabelas e exibe sua estrutura, permitindo verificar os nomes das colunas e seus respectivos tipos de dados.

In [0]:
for t in ["title_basics", "title_ratings", "title_principals", "name_basics", "title_episode"]:
    full_table = f"{CATALOG}.bronze.{t}"
    print(f"--- {full_table} ---")
    spark.table(full_table).printSchema()

--- workspace.bronze.title_basics ---
root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

--- workspace.bronze.title_ratings ---
root
 |-- tconst: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

--- workspace.bronze.title_principals ---
root
 |-- tconst: string (nullable = true)
 |-- ordering: string (nullable = true)
 |-- nconst: string (nullable = true)
 |-- category: string (nullable = true)
 |-- job: string

Esta célula exibe os 10 primeiros registros da tabela title_basics.

In [0]:
display(spark.table(f"{CATALOG}.bronze.title_basics").limit(10))

tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,_ingestion_timestamp,_source_file
tt0000001,short,Carmencita,Carmencita,0,1894,null,1,"Documentary,Short",2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,null,5,"Animation,Short",2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,null,5,"Animation,Comedy,Romance",2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000004,short,Un bon bock,Un bon bock,0,1892,null,12,"Animation,Short",2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,null,1,Short,2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000006,short,Chinese Opium Den,Chinese Opium Den,0,1894,null,1,Short,2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000007,short,Corbett and Courtney Before the Kinetograph,Corbett and Courtney Before the Kinetograph,0,1894,null,1,"Short,Sport",2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000008,short,Edison Kinetoscopic Record of a Sneeze,Edison Kinetoscopic Record of a Sneeze,0,1894,null,1,"Documentary,Short",2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,null,45,Romance,2026-09-27T14:00:40.077Z,title.basics.tsv.gz
tt0000010,short,Leaving the Factory,La sortie de l'usine Lumière à Lyon,0,1895,null,1,"Documentary,Short",2026-09-27T14:00:40.077Z,title.basics.tsv.gz
